In [2]:
import os

In [3]:
import numpy as np

In [4]:
from sklearn.preprocessing import LabelEncoder

In [5]:
from sklearn.neural_network import MLPClassifier

In [6]:
from imutils import paths

In [7]:
from PIL import Image

In [8]:
from numpy import asarray

In [9]:
from sklearn.preprocessing import LabelEncoder

In [10]:
from sklearn.model_selection import train_test_split

In [50]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [12]:
from tqdm import tqdm

In [13]:
import matplotlib.pyplot as plt

In [14]:
from sklearn.metrics import classification_report

In [15]:
from sklearn.utils.class_weight import compute_class_weight

In [16]:
from collections import Counter

In [17]:
from sklearn.linear_model import LogisticRegression

In [18]:
def get_features(model,X):
    hidden = X
    for i in range(len(model.coefs_)-1):
        hidden = np.dot(hidden,model.coefs_[i]) + model.intercepts_[i]
        hidden = np.maximum(hidden,0)
    return hidden
def Open_and_resize(image_path):
    image = Image.open(image_path).convert("RGB")
    resize_image = image.resize((64,64))
    data = asarray(resize_image)
    return data

def good_image(arr_image,cls=None):
    if cls=="Horse":
        return np.var(arr_image)>1000        
    return np.var(arr_image)>3000        
        
dataset = r'C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\PetImages'
image_list = os.listdir(dataset)
print(image_list)
X=[]
Y=[]
for i in image_list:
    folders_type = os.path.join(dataset, i)
    folder_name = os.path.basename(folders_type)
    print(folders_type)
    if "Cat" in str(folder_name):
        print("cat")
        for cat_img in tqdm(os.listdir(folders_type)):
             cat_image_path = os.path.join(folders_type, cat_img)
             try:
                     cat_arr=Open_and_resize(cat_image_path)
                     if cat_arr.shape ==(64,64,3) and good_image(cat_arr) :
                         cat_arr_float = cat_arr.flatten().astype('float32') / 255.0
                         X.append(cat_arr_float)
                         Y.append("Cat") 
             except (UnidentifiedImageError, OSError):
                    print("Can't open:", cat_image_path)
    elif "Dog" in str(folder_name):
         print("dog")
         for dog_img in tqdm(os.listdir(folders_type)):
             dog_image_path = os.path.join(folders_type,dog_img)
             try:
                     dog_arr=Open_and_resize(dog_image_path)
                     if dog_arr.shape ==(64,64,3) and good_image(dog_arr):
                         dog_arr_float = dog_arr.flatten().astype('float32')/255.0
                         X.append(dog_arr_float)
                         Y.append("Dog")
             except (UnidentifiedImageError, OSError):
                     print("Can't open:", dog_image_path)
X = np.array(X)

['Cat', 'Dog', 'Horse']
C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\PetImages\Cat
cat


100%|███████████████████████████████████████████████████████████████████████████| 12499/12499 [00:46<00:00, 266.62it/s]


C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\PetImages\Dog
dog


 91%|████████████████████████████████████████████████████████████████████▌      | 11427/12499 [00:55<00:05, 199.96it/s]C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\venv\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))
100%|███████████████████████████████████████████████████████████████████████████| 12499/12499 [01:01<00:00, 203.87it/s]


C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\PetImages\Horse


In [19]:
lable = LabelEncoder()
y_lable = lable.fit_transform(Y)

In [20]:
x_train,x_test,y_train,y_test=train_test_split(X,y_lable,test_size=0.2,random_state=42,stratify=y_lable)

In [21]:
MLPModel = MLPClassifier(
    hidden_layer_sizes=(64,32),
    activation='relu',
    solver='adam',
    max_iter=1000
)
MLPModel.fit(x_train,y_train)


,hidden_layer_sizes,"(64, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,1000
,shuffle,True
,random_state,None


In [41]:
y_predict = MLPModel.predict(x_test)

In [49]:
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(6038), np.int64(1): np.int64(5186)}


In [53]:
print("Accuracy:", accuracy_score(y_test, y_predict))
print(classification_report(y_test, y_predict))
print(confusion_matrix(y_test, y_predict))

Accuracy: 0.6140413399857448
              precision    recall  f1-score   support

           0       0.64      0.66      0.65      1510
           1       0.59      0.56      0.57      1296

    accuracy                           0.61      2806
   macro avg       0.61      0.61      0.61      2806
weighted avg       0.61      0.61      0.61      2806

[[1003  507]
 [ 576  720]]


In [42]:
X_base_features = get_features(MLPModel,X)
y_base_encoded = y_lable

In [43]:
X_horse=[]
Y_horse=[]
if "Horse" in str(folder_name):
    print("horse")
    for horse_img in tqdm(os.listdir(folders_type)):
        horse_image_path = os.path.join(folders_type,horse_img)
        try:
                horse_arr = Open_and_resize(horse_image_path)
                if horse_arr.shape == (64,64,3) and good_image(horse_arr,cls="Horse"):
                    horse_arr_float = horse_arr.flatten().astype('float32')/255.0
                    X_horse.append(horse_arr_float)
                    Y_horse.append("Horse")
        except(UnidentifiedImageError,OSError):
                print("Can't Open",horse_image_path)
X_horse = np.array(X_horse)

horse


100%|█████████████████████████████████████████████████████████████████████████████| 1170/1170 [00:03<00:00, 353.27it/s]


In [44]:
X_horse_features = get_features(MLPModel,X_horse)
y_horse_encoded = np.full(len(X_horse_features),2)

In [45]:
X_all_feauters = np.vstack([X_base_features,X_horse_features])
Y_all_encoded = np.concatenate([y_base_encoded,np.full(len(X_horse_features), 2)])

In [46]:
clf = LogisticRegression(max_iter=2000,multi_class='auto')
clf.fit(X_all_feauters,Y_all_encoded)

C:\Users\Ghazal\Documents\GitHub\Cats_and_dogs-CNN\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'auto'


In [47]:
def predict_image(image_path,MLPModel,clf):
    try:
        arr_image = Open_and_resize(image_path)
        if arr_image.shape != (64,64,3) and  good_image(arr_image):
            print("Invalid shape:", arr.shape)
            return None
        arr = arr_image.flatten().astype('float32') / 255.0
        arr = arr_image.reshape(1, -1)

        features = get_features(MLPModel, arr)

        pred_class = clf.predict(features)[0]
        print(pred_class)

        # cat=0 , dog=1 , horse=2
        if pred_class == 2:
            return "Horse"
        elif pred_class == 1:
            return "Dog"
        elif pred_class == 0:
            return "Cat"
            
    
    except Exception as e:
        print("Error:", e)
        return None


print(predict_image(r"C:\Users\Ghazal\Desktop\cat.jpg", MLPModel, clf))

1
Dog


In [48]:
report = classification_report(y_test,y_predict,target_names=['Cat','Dog'])
print(report)

              precision    recall  f1-score   support

         Cat       0.64      0.66      0.65      1510
         Dog       0.59      0.56      0.57      1296

    accuracy                           0.61      2806
   macro avg       0.61      0.61      0.61      2806
weighted avg       0.61      0.61      0.61      2806

